In [1]:
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path.cwd().parent
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "forward_model_raw.csv"
)

model_df = pd.read_csv(DATA_PATH)

print("Veri boyutu:", model_df.shape)
display(model_df.head())

Veri boyutu: (20372, 8)


,qmof_id,smiles_nodes,smiles_linkers,point_group,topology,density,pld,lcd
0,qmof-8a95c27,"['O', '[Ba]', '[Cu]']",['[O-]C=O'],-1,NaN,2.763246,0.68822,1.35480
1,qmof-019ba28,NaN,NaN,2/m,NaN,3.229952,1.18570,2.13507
2,qmof-830ed1c,['[Co]'],['[O-]C(=O)c1ccncc1'],m,rtl,1.557644,2.36128,4.21176
3,qmof-5bd4a24,['[Co]'],['[O-]C(=O)c1ccncc1'],2/m,rtl,1.616139,2.14542,3.27957
4,qmof-644aab4,['[Zn][Zn]'],"['[O-]C(=O)c1cccc(c1)c1nccs1', 'n1ccc(cc1)c1cc...",-1,NaN,1.596537,1.33452,2.03948


In [2]:
smiles_columns = [
    "smiles_nodes",
    "smiles_linkers",
]

categorical_columns = [
    "point_group",
    "topology",
]

target_columns = [
    "density",
    "pld",
    "lcd",
]

In [3]:
strict_df = model_df.dropna(
    subset=smiles_columns + categorical_columns
).copy()

print("Tamamen dolu veri seti:", strict_df.shape)

Tamamen dolu veri seti: (7899, 8)


In [4]:
main_df = model_df.dropna(
    subset=smiles_columns
).copy()

main_df["topology_missing"] = (
    main_df["topology"]
    .isna()
    .astype("int8")
)

main_df["topology"] = (
    main_df["topology"]
    .fillna("__MISSING_TOPOLOGY__")
)

print("Ana model veri seti:", main_df.shape)

print("\nKalan eksik değerler:")
print(main_df.isna().sum())

Ana model veri seti: (17544, 9)

Kalan eksik değerler:
qmof_id             0
smiles_nodes        0
smiles_linkers      0
point_group         0
topology            0
density             0
pld                 0
lcd                 0
topology_missing    0
dtype: int64


In [5]:
full_df = model_df.copy()

full_df["smiles_nodes_missing"] = (
    full_df["smiles_nodes"]
    .isna()
    .astype("int8")
)

full_df["smiles_linkers_missing"] = (
    full_df["smiles_linkers"]
    .isna()
    .astype("int8")
)

full_df["topology_missing"] = (
    full_df["topology"]
    .isna()
    .astype("int8")
)

full_df["smiles_nodes"] = (
    full_df["smiles_nodes"]
    .fillna("__MISSING_NODE_SMILES__")
)

full_df["smiles_linkers"] = (
    full_df["smiles_linkers"]
    .fillna("__MISSING_LINKER_SMILES__")
)

full_df["topology"] = (
    full_df["topology"]
    .fillna("__MISSING_TOPOLOGY__")
)

print("Bütün kayıtların tutulduğu veri seti:", full_df.shape)
print("\nKalan eksik değerler:")
print(full_df.isna().sum())

Bütün kayıtların tutulduğu veri seti: (20372, 11)

Kalan eksik değerler:
qmof_id                   0
smiles_nodes              0
smiles_linkers            0
point_group               0
topology                  0
density                   0
pld                       0
lcd                       0
smiles_nodes_missing      0
smiles_linkers_missing    0
topology_missing          0
dtype: int64


In [6]:
experiment_summary = pd.DataFrame({
    "dataset": [
        "strict_complete_case",
        "main_topology_imputed",
        "full_missing_tokens",
    ],
    "row_count": [
        len(strict_df),
        len(main_df),
        len(full_df),
    ],
})

experiment_summary["retained_percentage"] = (
    experiment_summary["row_count"]
    / len(model_df)
    * 100
).round(2)

display(experiment_summary)

,dataset,row_count,retained_percentage
0,strict_complete_case,7899,38.77
1,main_topology_imputed,17544,86.12
2,full_missing_tokens,20372,100.00


In [7]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

strict_df.to_csv(
    PROCESSED_DIR / "forward_model_strict.csv",
    index=False,
)

main_df.to_csv(
    PROCESSED_DIR / "forward_model_main.csv",
    index=False,
)

full_df.to_csv(
    PROCESSED_DIR / "forward_model_full.csv",
    index=False,
)

print("Üç veri seti kaydedildi.")

Üç veri seti kaydedildi.
